# Лабораторная 1. Интерактивный анализ данных велопарковок SF Bay Area Bike Share в Apache Spark

## Описание данных

https://www.kaggle.com/benhamner/sf-bay-area-bike-share

stations.csv схема:

```
id: station ID number
name: name of station
lat: latitude
long: longitude
dock_count: number of total docks at station
city: city (San Francisco, Redwood City, Palo Alto, Mountain View, San Jose)
installation_date: original date that station was installed. If station was moved, it is noted below.
```

trips.csv схема:

```
id: numeric ID of bike trip
duration: time of trip in seconds
start_date: start date of trip with date and time, in PST
start_station_name: station name of start station
start_station_id: numeric reference for start station
end_date: end date of trip with date and time, in PST
end_station_name: station name for end station
end_station_id: numeric reference for end station
bike_id: ID of bike used
subscription_type: Subscriber = annual or 30-day member; Customer = 24-hour or 3-day member
zip_code: Home zip code of subscriber (customers can choose to manually enter zip at kiosk however data is unreliable)
```

## Подключение Spark

In [1]:
from pyspark.sql import SparkSession
from pyspark import SparkContext, SparkConf
from math import radians, sin, cos, sqrt, atan2
from datetime import datetime

conf = SparkConf().setAppName("Lab1_PySpark").setMaster("local[*]")
sc = SparkContext.getOrCreate(conf=conf)

spark = SparkSession.builder \
    .appName("Lab1_PySpark") \
    .master("local[*]") \
    .getOrCreate()

print(f"Spark version: {sc.version}")

Spark version: 4.0.2


## Загрузка данных

In [2]:
tripData = sc.textFile("trips.csv")
tripsHeader = tripData.first()
trips = tripData.filter(lambda row: row != tripsHeader).map(lambda row: row.split(",", -1))

stationData = sc.textFile("stations.csv")
stationsHeader = stationData.first()
stations = stationData.filter(lambda row: row != stationsHeader).map(lambda row: row.split(",", -1))

## Подготовка данных

In [3]:
def parse_station(row):
    try:
        return {
            'station_id': int(row[0]),
            'name': row[1],
            'lat': float(row[2]),
            'long': float(row[3]),
            'dockcount': int(row[4]),
            'landmark': row[5],
            'installation': datetime.strptime(row[6], '%m/%d/%Y')
        }
    except:
        return None

def parse_trip(row):
    try:
        return {
            'trip_id': int(row[0]),
            'duration': int(row[1]),
            'start_date': datetime.strptime(row[2], '%m/%d/%Y %H:%M'),
            'start_station_name': row[3],
            'start_station_id': int(row[4]),
            'end_date': datetime.strptime(row[5], '%m/%d/%Y %H:%M'),
            'end_station_name': row[6],
            'end_station_id': int(row[7]),
            'bike_id': int(row[8]),
            'subscription_type': row[9],
            'zip_code': row[10]
        }
    except:
        return None

In [4]:
stationsInternal = stations.map(parse_station).filter(lambda x: x is not None)
tripsInternal = trips.map(parse_trip).filter(lambda x: x is not None)

print(f"Станций: {stationsInternal.count()}, Поездок: {tripsInternal.count()}")

Станций: 70, Поездок: 669957


## Решение задач

### 1. Найти велосипед с максимальным временем пробега.

In [5]:
bike_duration = tripsInternal.map(lambda t: (t['bike_id'], t['duration'])) \
                              .reduceByKey(lambda a, b: a + b)

max_bike_id, max_duration = bike_duration.takeOrdered(1, key=lambda x: -x[1])[0]
print(f"Ответ: Велосипед с ID {max_bike_id}")
print(f"Время пробега: {max_duration} секунд ({max_duration/3600:.2f} часов)")

Ответ: Велосипед с ID 535
Время пробега: 18611693 секунд (5169.91 часов)


### 2. Найти наибольшее геодезическое расстояние между станциями.

In [6]:
def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = sin(dlat / 2) ** 2 + cos(lat1) * cos(lat2) * sin(dlon / 2) ** 2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))
    return R * c

stations_coords = stationsInternal.map(lambda s: (s['station_id'], s['lat'], s['long'], s['name']))

# Декартово произведение всех станций, исключаем повторяющиеся пары
max_station = stations_coords.cartesian(stations_coords) \
    .filter(lambda x: x[0][0] < x[1][0]) \
    .map(lambda x: (x[0][0], x[0][3], x[1][0], x[1][3],
                    haversine(x[0][1], x[0][2], x[1][1], x[1][2]))) \
    .takeOrdered(1, key=lambda x: -x[4])[0]

print(f"Ответ: {max_station[4]:.2f} км между станциями {max_station[1]} и {max_station[3]}")

Ответ: 69.92 км между станциями SJSU - San Salvador at 9th и Embarcadero at Sansome


### 3. Найти путь велосипеда с максимальным временем пробега через станции.

In [7]:
bike_trips = tripsInternal.filter(lambda t: t['bike_id'] == max_bike_id) \
                          .map(lambda t: (t['duration'], t)) \
                          .sortByKey(ascending=False) \
                          .map(lambda x: x[1])

total_trips = tripsInternal.filter(lambda t: t['bike_id'] == max_bike_id).count()
print(f"Всего поездок у велосипеда {max_bike_id}: {total_trips}")

print(f"\nПервые 10 поездок велосипеда {max_bike_id}:")
print(f"{'':<5} {'Длительность, мин':<20} {'Станция старта':<30} {'Станция финиша':<30}")
print("-" * 85)

for idx, trip in enumerate(bike_trips.take(10), 1):
    duration_min = trip['duration'] / 60
    start = trip['start_station_name'][:27] + ".." if len(trip['start_station_name']) > 29 else trip['start_station_name']
    end = trip['end_station_name'][:27] + ".." if len(trip['end_station_name']) > 29 else trip['end_station_name']
    print(f"{idx:<5} {duration_min:<20.1f} {start:<30} {end:<30}")


Всего поездок у велосипеда 535: 1328

Первые 10 поездок велосипеда 535:
      Длительность, мин    Станция старта                 Станция финиша                
-------------------------------------------------------------------------------------
1     287840.0             South Van Ness at Market       2nd at Folsom                 
2     1460.6               Powell Street BART             Civic Center BART (7th at M.. 
3     561.0                San Francisco Caltrain (Tow..  Steuart at Market             
4     431.8                Market at 10th                 San Francisco Caltrain (Tow.. 
5     419.6                2nd at Folsom                  Mechanics Plaza (Market at .. 
6     415.3                Powell at Post (Union Square)  Powell at Post (Union Square) 
7     379.8                Market at Sansome              Market at Sansome             
8     372.7                Broadway St at Battery St      Harry Bridges Plaza (Ferry .. 
9     352.6                Grant Avenue a

### 4. Найти количество велосипедов в системе.

In [8]:
unique_bikes = tripsInternal.map(lambda t: t['bike_id']).distinct()
bike_count = unique_bikes.count()
print(f"Ответ: {bike_count} велосипедов")

Ответ: 700 велосипедов


### 5. Найти пользователей потративших на поездки более 3 часов.

In [9]:
THREE_HOURS_SEC = 3 * 3600

user_time = tripsInternal.filter(lambda t: t.get('zip_code')) \
                         .map(lambda t: (t['zip_code'], t['duration'])) \
                         .reduceByKey(lambda a, b: a + b)

heavy_users = user_time.filter(lambda x: x[1] > THREE_HOURS_SEC)
heavy_users_count = heavy_users.count()

print(f"Ответ: Найдено {heavy_users_count} пользователей потративших на поездки более 3 часов")

print("\nТоп-10 пользователей с наибольшим временем:")
top_users = heavy_users.takeOrdered(10, key=lambda x: -x[1])

print(f"{'Zip code':<10} {'Время поездок':>15} {'Число поездок':>15}")
print("-" * 45)

for zip_code, total_sec in top_users:
    trip_count = tripsInternal.filter(lambda t: t.get('zip_code') == zip_code).count()
    print(f"{zip_code:<10} {total_sec/3600:>15.2f} {trip_count:>15,}")

Ответ: Найдено 3660 пользователей потративших на поездки более 3 часов

Топ-10 пользователей с наибольшим временем:
Zip code     Время поездок   Число поездок
---------------------------------------------
94107             13821.43          78,704
nil               12701.54          10,682
94105              7110.04          42,672
94133              6010.47          31,359
94102              5313.34          19,757
94103              5313.16          26,673
95531              4797.33               1
94111              3956.94          21,409
95112              3539.55          11,564
94109              3349.20          13,989


In [10]:
sc.stop()